# ETL MV Agent-Only 全量 SQL 实验流程

本 notebook 用于串起 `llm_demo` 的 agent-only 原型。当前 Executor 仍是 dry-run，只输出 MV 物化状态和 SQL 运行顺序，不连接 Spark。

In [1]:
from pathlib import Path
import sys
from dotenv import load_dotenv

# 读取项目根目录 .env，并把项目根目录加入 Python import 路径。
# 这样即使从 llm_demo/ 或 llm_demo/notebooks 启动 Jupyter，也能 import llm_demo。
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in [cwd, *cwd.parents]
    if (path / "llm_demo").is_dir() and (path / "tpcds-spark").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")
PROJECT_ROOT

PosixPath('/Users/zlhmac/code/Python_project/MV_gen')

In [2]:
from llm_demo.src.agents.sql_loader_agent import SQLLoaderAgent
from llm_demo.src.agents.feature_agent import FeatureAgent
from llm_demo.src.agents.family_agent import FamilyAgent
from llm_demo.src.agents.batch_cluster_agent import BatchClusterAgent
from llm_demo.src.agents.self_iteration_agent import SelfIterationAgent
from llm_demo.src.core.artifact_store import ArtifactStore
from llm_demo.src.core.llm_client import LLMClient
from llm_demo.src.core.query_analysis_runner import QueryAnalysisRunner
from llm_demo.src.core.family_candidate_builder import FamilyCandidateBuilder
from llm_demo.src.core.batch_workflow_runner import BatchWorkflowRunner
from llm_demo.src.core.coverage_summary_builder import CoverageSummaryBuilder

In [3]:
from datetime import datetime

# 每次实验使用独立 run_id；失败排查通过 artifacts 和 run_log 查看。
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
store = ArtifactStore(run_id=run_id, artifact_root=PROJECT_ROOT / "llm_demo" / "artifacts")
llm_client = LLMClient(project_root=PROJECT_ROOT)
run_id

'20260612_151048'

In [4]:
# 1. 读取完整 tpcds-spark 目录，生成 sql_manifest.json。
#    SQLLoader 只记录 SQL 文件路径，不复制 SQL Text，避免 artifact 目录膨胀。
sql_manifest_path = SQLLoaderAgent(store).run(PROJECT_ROOT / "tpcds-spark")
sql_manifest_path

PosixPath('/Users/zlhmac/code/Python_project/MV_gen/llm_demo/artifacts/20260612_151048/00_raw_sql/sql_manifest.json')

In [5]:
# 2. 串行 Feature intake：每次只把一条 SQL 放入 LLM prompt。
#    首轮失败会在全部 SQL 处理结束后统一 retry 一次；resume_feature=True 只跳过当前 run 已成功项。
feature_agent = FeatureAgent(store, llm_client)
query_blocks_path = QueryAnalysisRunner(store, feature_agent).run_all(sql_manifest_path, resume_feature=False)
query_blocks_path

PosixPath('/Users/zlhmac/code/Python_project/MV_gen/llm_demo/artifacts/20260612_151048/01_query_blocks/query_blocks.json')

In [6]:
# 3. 人工查看 Feature 覆盖情况。
feature_status = store.read_json(store.feature_status_path)
feature_status["queries"][:5]

[{'query_id': 'q01',
  'status': 'success',
  'attempts': 1,
  'qb_ids': ['q01.outer'],
  'unsupported_reasons': [],
  'error_type': None,
  'error_message': None},
 {'query_id': 'q02',
  'status': 'success',
  'attempts': 1,
  'qb_ids': ['q02.outer'],
  'unsupported_reasons': [],
  'error_type': None,
  'error_message': None},
 {'query_id': 'q1',
  'status': 'success',
  'attempts': 1,
  'qb_ids': ['q1.cte_1', 'q1.outer', 'q1.subquery_1'],
  'unsupported_reasons': [],
  'error_type': None,
  'error_message': None},
 {'query_id': 'q10',
  'status': 'success',
  'attempts': 1,
  'qb_ids': ['q10.outer',
   'q10.subquery_1',
   'q10.subquery_2',
   'q10.subquery_3'],
  'unsupported_reasons': [],
  'error_type': None,
  'error_message': None},
 {'query_id': 'q11',
  'status': 'partial_success',
  'attempts': 1,
  'qb_ids': ['q11.outer', 'q11.cte_1', 'q11.branch_1', 'q11.branch_2'],
  'unsupported_reasons': ['outer block references CTE year_total only; no physical tables available to popula

In [7]:
# 4. 先用 code-only builder 压缩 family 候选空间，再由 FamilyAgent 做 LLM evaluate / merge / split。
family_candidates_path = FamilyCandidateBuilder(store).run(query_blocks_path)
families_path = FamilyAgent(store, llm_client).run(family_candidates_path, query_blocks_path=query_blocks_path)
families_path

PosixPath('/Users/zlhmac/code/Python_project/MV_gen/llm_demo/artifacts/20260612_151048/02_families/query_families.json')

In [8]:
# 5. 按 batch_classification.csv 生成 global batch；CSV 是当前分 batch 的 source of truth。
classification_csv_path = PROJECT_ROOT / "batch_classification.csv"

batches_path = BatchClusterAgent(store, llm_client).run_from_classification_csv(
    classification_csv_path=classification_csv_path,
    sql_manifest_path=sql_manifest_path,
    query_blocks_path=query_blocks_path,
    families_path=families_path,
)
batches_path

PosixPath('/Users/zlhmac/code/Python_project/MV_gen/llm_demo/artifacts/20260612_151048/03_batches/complexity_batches.json')

In [9]:
# 6. 人工查看 CSV 分 batch 结果和 batch 内 family_groups 覆盖情况。
batches = store.read_json(batches_path)["complexity_batches"]
[
    {
        "batch_id": batch["batch_id"],
        "batch_type": batch["batch_type"],
        "query_count": len(batch.get("query_ids", [])),
        "family_group_count": len(batch.get("family_groups", [])),
    }
    for batch in batches
]

[{'batch_id': 1,
  'batch_type': 'Batch-1',
  'query_count': 15,
  'family_group_count': 6},
 {'batch_id': 2,
  'batch_type': 'Batch-2',
  'query_count': 29,
  'family_group_count': 11},
 {'batch_id': 3,
  'batch_type': 'Batch-3',
  'query_count': 16,
  'family_group_count': 4},
 {'batch_id': 4,
  'batch_type': 'Batch-4',
  'query_count': 17,
  'family_group_count': 4},
 {'batch_id': 5,
  'batch_type': 'Batch-5',
  'query_count': 27,
  'family_group_count': 11},
 {'batch_id': 6,
  'batch_type': 'Other',
  'query_count': 1,
  'family_group_count': 0}]

In [10]:
# 7. 对每个非空 batch 执行 dry-run 闭环：historical rewrite -> batch-local MV -> materialize state -> final rewrite -> query order。
#    Executor 当前不运行 Spark，只维护 materialized_mvs.json 和 execution_order.json。
workflow_outputs = BatchWorkflowRunner(store, llm_client).run_all_batches(
    complexity_batches_path=batches_path,
    sql_manifest_path=sql_manifest_path,
    query_blocks_path=query_blocks_path,
    families_path=families_path,
)
workflow_outputs[-5:]

ValueError: Rewrite output query_ids {'q13', 'q46', 'q61', 'q34', 'q6', 'q27', 'q28', 'q18', 'q32', 'q45', 'q68', 'q9', 'q88', 'q42', 'q73', 'q79', 'q92', 'q39b', 'q90', 'q21', 'q48', 'q53', 'q98', 'q65', 'q52', 'q89', 'q12', 'q20', 'q59', 'q39a', 'q63'} do not match expected {'q13', 'q46', 'q61', 'q34', 'q6', 'q27', 'q28', 'q18', 'q32', 'q45', 'q68', 'q9', 'q88', 'q73', 'q79', 'q92', 'q39b', 'q90', 'q21', 'q48', 'q53', 'q98', 'q65', 'q89', 'q12', 'q20', 'q59', 'q39a', 'q63'}

In [ ]:
# 8. 生成覆盖率摘要。coverage_summary 不参与业务决策，只帮助人工判断本轮实验质量。
coverage_path = CoverageSummaryBuilder(store).run()
store.read_json(coverage_path)

In [ ]:
# 9. 基于 run log 和相关 artifacts 生成 rules 反馈建议；不会自动修改 rules 文件。
feedback_path = SelfIterationAgent(store, llm_client).run(store.run_log_path)
store.read_json(feedback_path)